# Problem 39: Scoped Memory

**Difficulty:** Medium | **Framework:** OpenAI Agents SDK

**Categories:** memory, orchestration

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app/problems/scoped-memory).

## 1. Install Dependencies

In [ ]:
!pip install -q openai-agents

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- Each agent must have its own private memory scope
- A shared scope must exist for cross-agent communication
- Private memory of agent A must never be visible to agent B
- The shared scope should only contain explicitly published information


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from agents import Agent, Runner

# BUG: All agents share a single global memory — no scoping
global_memory = []

def make_researcher():
    def run_researcher(task: str) -> str:
        global_memory.append("[Researcher internal] Raw data contains PII: user SSN 123-45-6789")
        global_memory.append("[Researcher] Found 3 relevant articles about AI safety.")
        context = "\n".join(global_memory)
        agent = Agent(
            name="Researcher",
            instructions=f"You are a researcher. Memory:\n{context}",
        )
        result = Runner.run_sync(agent, task)
        return result.final_output
    return run_researcher

def make_writer():
    def run_writer(research: str) -> str:
        global_memory.append("[Writer internal] Draft has grammatical issues, fixing...")
        global_memory.append("[Writer] Completed first draft of the report.")
        # BUG: Writer can see researcher's private notes including PII
        context = "\n".join(global_memory)
        agent = Agent(
            name="Writer",
            instructions=f"You are a writer. Memory:\n{context}",
        )
        result = Runner.run_sync(agent, f"Write a report based on: {research}")
        return result.final_output
    return run_writer

researcher = make_researcher()
writer = make_writer()

research_output = researcher("Research AI safety trends.")
final_report = writer(research_output)
print(final_report)
print(f"\nGlobal memory (visible to all): {global_memory}")

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Use a dictionary with namespace keys (e.g., 'agent_a_private', 'agent_b_private', 'shared') to separate memory scopes
2. Each agent should only read from its own private scope and the shared scope
3. Create helper functions like write_private(), write_shared(), and read_context() that enforce scope boundaries


## 7. Evaluation Criteria

Check your solution against these criteria:

- Private memory is isolated
- Shared scope enables collaboration
- Sensitive data stays private
